In [3]:
import pandas as pd
import numpy as np

# ===== 경로 및 기본 설정 =====
PATH = "2014_2020_train_full_interpolated_filled_lag_plus.csv"  # 필요 시 수정
datetime_col = "ymd"          # 시간 컬럼명 (다르면 여기 수정)
site_col = "code_new"         # 지점 ID 컬럼명
target_col = "elev"           # 지하수 수위 컬럼명

# ===== 데이터 로드 & 정렬 =====
df = pd.read_csv(PATH, encoding='cp949')
# 시간 컬럼 파싱
df[datetime_col] = pd.to_datetime(df[datetime_col])
# 지점-시간 기준 정렬
df = df.sort_values([site_col, datetime_col]).reset_index(drop=True)

# 원본 보존용 복사본
out = df.copy()

# ===== 그룹별 보조 시리즈 =====
g = out.groupby(site_col, sort=False)
first_elev = g[target_col].transform("first")  # 지점별 첫 elev
pos = g.cumcount()                             # 지점별 0,1,2,... 인덱스(행 순번)

def make_lag(k_hours: int):
    """
    k시간 래그 컬럼과 초기 프리필(mask) 인덱스 컬럼 생성.
    - 래그: 지점별 shift(k)
    - 초기 k행은 지점의 첫 elev로 채움
    - 프리필 인덱스: 초기 k행만 1, 나머지 0 (int8)
    """
    lag_col = f"{target_col}_lag_{k_hours}h"
    idx_col = f"is_prefill_{k_hours}h"

    # 1) 기본 래그(지점별)
    out[lag_col] = g[target_col].shift(k_hours)

    # 2) 초기 k행 마스크(지점별 첫 k개 행)
    mask = pos < k_hours

    # 3) 초기 구간은 지점별 '첫 elev'로 프리필
    out.loc[mask, lag_col] = first_elev[mask].values

    # 4) 프리필 인덱스(0/1)
    out[idx_col] = mask.astype("int8")

    return lag_col, idx_col

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 736416 entries, 0 to 736415
Data columns (total 13 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   ymd                       736416 non-null  datetime64[ns]
 1   code_new                  736416 non-null  int64         
 2   elev                      736416 non-null  float64       
 3   wtemp                     736416 non-null  float64       
 4   ec                        736416 non-null  int64         
 5   기온(°C)                    736416 non-null  float64       
 6   강수량(mm)                   736416 non-null  float64       
 7   풍속(m/s)                   736416 non-null  float64       
 8   습도(%)                     736416 non-null  int64         
 9   현지기압(hPa)                 736416 non-null  float64       
 10  지면온도(°C)                  736416 non-null  float64       
 11  is_precipitation_missing  736416 non-null  int64         
 12  el

In [5]:
# ===== 요구된 3개 래그 생성 =====
cols = []
for k in (336, 504, 720):  # 14일, 21일, 30일 (시간 단위)
    cols.extend(make_lag(k))

In [9]:
# 확인용(처음 몇 행)
print(out[[site_col, datetime_col] + cols].head(20))

    code_new                 ymd  elev_lag_336h  is_prefill_336h  \
0          1 2014-01-01 00:00:00         105.47                1   
1          1 2014-01-01 01:00:00         105.47                1   
2          1 2014-01-01 02:00:00         105.47                1   
3          1 2014-01-01 03:00:00         105.47                1   
4          1 2014-01-01 04:00:00         105.47                1   
5          1 2014-01-01 05:00:00         105.47                1   
6          1 2014-01-01 06:00:00         105.47                1   
7          1 2014-01-01 07:00:00         105.47                1   
8          1 2014-01-01 08:00:00         105.47                1   
9          1 2014-01-01 09:00:00         105.47                1   
10         1 2014-01-01 10:00:00         105.47                1   
11         1 2014-01-01 11:00:00         105.47                1   
12         1 2014-01-01 12:00:00         105.47                1   
13         1 2014-01-01 13:00:00         105.47 

In [10]:
out.to_csv(PATH.replace(".csv", "_with_elev_lags.csv"), index=False, encoding="cp949")

In [9]:
import pandas as pd

# ===== 경로 & 컬럼명 설정 =====
PATH = "2014_2020_train_full_interpolated_filled_lag_plus_with_elev_lags.csv"  # 필요 시 수정
datetime_col = "ymd"          # 시간 컬럼명
site_col = "code_new"         # 지점 ID 컬럼명
rain_col = "강수량(mm)"        # 강수량 컬럼명 (다르면 여기 수정)

# ===== 로드 & 정렬 =====
df = pd.read_csv(PATH, encoding='cp949')
df[datetime_col] = pd.to_datetime(df[datetime_col])
df = df.sort_values([site_col, datetime_col]).reset_index(drop=True)

out = df.copy()
g = out.groupby(site_col, sort=False)

# 지점별 행 순번(0,1,2,...) : 첫 24시간(0~23) 마스크용
pos = g.cumcount()

# ===== 24시간 누적 강수량 =====
# 첫 24시간 구간은 00시부터의 누적합, 이후는 직전 24시간 롤링합
out["sum_of_rain_24h"] = (
    g[rain_col]
    .rolling(window=24, min_periods=1)   # 첫날(0~23시)은 가용 구간 누적
    .sum()
    .reset_index(level=0, drop=True)
    .round(1)
)

# ===== 첫 24시간 마스크(0/1) =====
out["is_first24h_sum_24h"] = (pos < 24).astype("int64")

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 736416 entries, 0 to 736415
Data columns (total 20 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   ymd                       736416 non-null  datetime64[ns]
 1   code_new                  736416 non-null  int64         
 2   elev                      736416 non-null  float64       
 3   wtemp                     736416 non-null  float64       
 4   ec                        736416 non-null  int64         
 5   기온(°C)                    736416 non-null  float64       
 6   강수량(mm)                   736416 non-null  float64       
 7   풍속(m/s)                   736416 non-null  float64       
 8   습도(%)                     736416 non-null  int64         
 9   현지기압(hPa)                 736416 non-null  float64       
 10  지면온도(°C)                  736416 non-null  float64       
 11  is_precipitation_missing  736416 non-null  int64         
 12  el

In [11]:
# 확인
print(out[[site_col, datetime_col, "sum_of_rain_24h", "is_first24h_sum_24h"]].head(30))

    code_new                 ymd  sum_of_rain_24h  is_first24h_sum_24h
0          1 2014-01-01 00:00:00              0.0                    1
1          1 2014-01-01 01:00:00              0.0                    1
2          1 2014-01-01 02:00:00              0.0                    1
3          1 2014-01-01 03:00:00              0.0                    1
4          1 2014-01-01 04:00:00              0.0                    1
5          1 2014-01-01 05:00:00              0.0                    1
6          1 2014-01-01 06:00:00              0.0                    1
7          1 2014-01-01 07:00:00              0.0                    1
8          1 2014-01-01 08:00:00              0.0                    1
9          1 2014-01-01 09:00:00              0.0                    1
10         1 2014-01-01 10:00:00              0.0                    1
11         1 2014-01-01 11:00:00              0.0                    1
12         1 2014-01-01 12:00:00              0.0                    1
13    

In [12]:
out.to_csv(PATH.replace(".csv", "_with_sum24h.csv"), index=False, encoding="cp949")